#Sentiment Analysis using NLP Pipeline & ML Models


### Dataset: Twitter Sentiment Analysis Dataset

### Objective:
To build an end-to-end sentiment analysis system using NLP preprocessing, feature engineering, and ML models.

## 1. Data Understanding

In [15]:
#Load Dataset
import pandas as pd

df = pd.read_csv("twitter.csv")
print("Shape:", df.shape)

# First rows
df.head()

Shape: (31962, 3)


,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [17]:
#Check Columns
print(df.columns)


Index(['id', 'label', 'tweet'], dtype='object')


In [19]:
#Class Distribution
print("Class Distribution:")
print(df['label'].value_counts())

Class Distribution:
label
0    29720
1     2242
Name: count, dtype: int64


In [20]:
#Sample Tweet
print("Sample Tweet:")
print(df['tweet'][0])

Sample Tweet:
 @user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run


##NLP Preprocessing

In [21]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [22]:
def preprocess_text(text):

    if pd.isnull(text):
        return ""

    text = str(text).lower()

    # Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags (#word → word)
    text = re.sub(r'#', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    words = text.split()

    # Remove stopwords + lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

In [23]:
df['clean_text'] = df['tweet'].apply(preprocess_text)

df[['tweet', 'clean_text']].head()

,tweet,clean_text
0,@user when a father is dysfunctional and is s...,father dysfunctional selfish drag kid dysfunct...
1,@user @user thanks for #lyft credit i can't us...,thanks lyft credit cant use cause dont offer w...
2,bihday your majesty,bihday majesty
3,#model i love u take with u all the time in ...,model love u take u time urð± ðððð...
4,factsguide: society now #motivation,factsguide society motivation


##Feature Engineering

TF-IDF

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['clean_text'])

In [26]:
y = df['label']

##Model Building

Train-Test Split

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Logistic Regression

In [45]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)

#TF-IDF
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

Naive Bayes

In [46]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

Decision Tree

In [31]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

Random Forest

In [53]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)


XGBoost

In [55]:
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='logloss')

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

##Model Evaluation

In [56]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred, name):
    print(f"\n{name}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))

Results

In [57]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")
evaluate(y_test, y_pred_rf, "Random Forest")
evaluate(y_test, y_pred_xgb, "XGBoost")


Logistic Regression
Accuracy : 0.9516658845612389
Precision: 0.8972972972972973
Recall   : 0.36403508771929827
F1 Score : 0.5179407176287052

Naive Bayes
Accuracy : 0.9515094634756766
Precision: 0.9244186046511628
Recall   : 0.34868421052631576
F1 Score : 0.5063694267515924

Decision Tree
Accuracy : 0.9471296730799311
Precision: 0.6460396039603961
Recall   : 0.5723684210526315
F1 Score : 0.6069767441860465

Random Forest
Accuracy : 0.9569842014703582
Precision: 0.7928802588996764
Recall   : 0.5372807017543859
F1 Score : 0.6405228758169934

XGBoost
Accuracy : 0.9501016737056155
Precision: 0.801762114537445
Recall   : 0.3991228070175439
F1 Score : 0.5329428989751098


##Comparison

In [59]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ]
})

print(results)

                 Model  Accuracy
0  Logistic Regression  0.951666
1          Naive Bayes  0.951509
2        Decision Tree  0.947130
3        Random Forest  0.956984
4              XGBoost  0.950102


## Final Insights

1. Logistic Regression and Naive Bayes achieved high accuracy (~95%), but their recall and F1-score are low. This indicates that the models are biased towards the majority class.

2. The dataset is imbalanced, which leads to misleading accuracy. Therefore, accuracy alone is not a reliable metric for evaluation.

3. Decision Tree showed better recall compared to Logistic Regression and Naive Bayes, indicating improved detection of the minority class.

4. Random Forest achieved the highest accuracy (95.69%) and a better F1-score (0.64), making it the best performing model overall. It provides a good balance between precision and recall.

5. XGBoost also performed well, but its recall is lower compared to Random Forest, resulting in a lower F1-score.

6. Among all models, Random Forest provides the best trade-off between accuracy and F1-score.

## Key Learnings

- Preprocessing steps like removing noise, stopwords, and lemmatization significantly improved model performance.
- TF-IDF proved to be an effective feature engineering technique for text classification.
- Accuracy is not sufficient for imbalanced datasets; F1-score is a better evaluation metric.
- Ensemble models (Random Forest, XGBoost) perform better than individual models.

## Best Model

Based on overall performance:
- Best Accuracy: Random Forest
- Best Balance (Precision + Recall): Random Forest

Hence, Random Forest is selected as the final model for this sentiment analysis task.